# Copy-wallet exposure signal



- **§18** — exposure-filter evaluation **80/50/20/10**: roi, total traded notional, net PnL, Sharpe on val & test (incl. Iran-excluded control)

(`position * price`) is a conviction/edge signal — copying only the wallet's high-exposure trades should

raise ROI, Sharpe, and profit-per-unit-vol.



This is the **exposure-focused** version of the notebook (the earlier `sig_val_opp_flipper` walkthrough,

composite, and capital-constrained sizing branches were removed as dead ends). §15 reads saved CSVs;

§16-18 recompute in-kernel from `cached_splits_exposure/*.parquet` (thresholds fit on train only).



Structure:

- **§1** — load raw Politics trades, split chronologically

- **§2** — per-wallet metrics

- **§14** — exposure → PnL/roi IC panel (within-wallet construction)

- **§15** — exposure-threshold sizing ladder (read-only from saved CSVs)

- **§16** — event-dominance & ROI diagnostics, recomputed in-kernel

- **§17** — volatility-neutral target `z_vol` (profit per unit vol)

- **§18** — exposure-filter evaluation **80/50/20/10**: roi, total traded notional, net PnL, Sharpe on val & test (incl. Iran-excluded control)


In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

NOTEBOOK_DIR = Path.cwd().resolve()
ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'signal_lab' else NOTEBOOK_DIR
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lib import (
    DEFAULT_TAGS,
    compute_copyable_notional,
    compute_opening_metrics,
    load_trades,
    split_data,
)
from polymarket_analysis.wallet_selection.volatility import compute_wallet_metrics
from signal_lab.signal_engines import compute_hold_time_metrics

## 1. Load raw trades and split them

In [3]:
df_full = load_trades(tags=DEFAULT_TAGS)
df_full = compute_copyable_notional(df_full)
df_train, df_val, df_test = split_data(df_full, method='chronological')

print(f'df_full={len(df_full):,} train={len(df_train):,} val={len(df_val):,} test={len(df_test):,}')


Markets: 3133887
Filtered markets for {'Politics'}: 48946
Loading 16 trade shards...
Total trades loaded: 14,657,642
Unique wallets: 35,688
Date range: 2025-01-01 00:00:59+00:00 -> 2026-09-07 17:39:54+00:00
Chronological split: train <= 2025-10-01T00:00:00Z, val <= 2026-04-08T00:00:00Z, test > 2026-04-08T00:00:00Z
Method: chronological  |  Unique end dates: 555  (train=222, val=166, test=167)

  Train:  1,688,748 trades  (3,651 markets)
  Val:    5,611,456 trades  (7,232 markets)
  Test:   7,357,438 trades  (13,391 markets)
  Total: 14,657,642 trades  (24,274 markets)
df_full=14,657,642 train=1,688,748 val=5,611,456 test=7,357,438


## 2. Compute wallet metrics directly in the notebook

In [4]:
wallet_metrics, _ = compute_wallet_metrics(df_train)
wallet_metrics['copyable_pnl_factor'] = np.clip(
    wallet_metrics['copyable_pnl'] / wallet_metrics['total_pnl'].replace(0, np.nan),
    0,
    1.0,
).fillna(0.0)
wallet_metrics['copyable_roi'] = wallet_metrics['average_roi'] * wallet_metrics['copyable_pnl_factor']

opening_metrics = compute_opening_metrics(df_train)
wallet_metrics = wallet_metrics.merge(opening_metrics, on='wallet', how='left')
for col in ['opening_roi', 'opening_pnl', 'opening_copyable_roi', 'opening_copyable_pnl']:
    wallet_metrics[col] = wallet_metrics[col].fillna(0.0)

hold_metrics = compute_hold_time_metrics(df_train)

display(wallet_metrics[['wallet', 'buy_roi', 'copyable_roi', 'trade_count', 'num_markets', 'num_buckets']].head())


,wallet,buy_roi,copyable_roi,trade_count,num_markets,num_buckets
0,0x00090e8b4fa8f88dc9c1740e460dd0f670021d43,0.079448,-0.002640,47,4,41
1,0x000b88e5ff8880d41f87070a7bd8bab414220872,0.399841,0.206344,52,24,47
2,0x000d257d2dc7616feaef4ae0f14600fdf50a758e,0.019414,-0.011883,2632,217,2012
3,0x000da5c4606f0c03bbe4dcbafc4458bdd10e54e0,0.272280,0.096182,106,14,91
4,0x002a380091a15a37d0ea7e144db922b6b6899a00,0.195168,0.001168,45,15,38


## 14. Copy-wallet exposure -> PnL IC

Test the hypothesis that a copy wallet's **post-trade exposure** (`position * price`) predicts its copyable edge.  Because cross-sectional exposure is dominated by wallet size, the primary axis is **wallet-relative**:

- `sig_exp_wt_rank` — within-wallet exposure rank in [0, 1] (train-fit)
- `sig_exp_wt_minmax` — within-wallet min-max in [0, 1] (train-fit, diagnostic)
- `sig_exp_marg` / `sig_exp_marg_wt` — marginal exposure `quantity*price` (and its wallet-relative rank): the book *added* by this trade, the averaging-down contrast
- `sig_exp_thr_{0.8,0.5,0.2}` — per-wallet thresholds capturing 100/80/50/20% of train copyable PnL
- `sig_exp_top_{0.8,0.5,0.2}` — per-wallet thresholds at the train exposure quantile `1 - t` (top 80/50/20% of the wallet's own book)

IC is measured against **`copyable_pnl`** (dollar edge) and **`roi_res`** (price-residualized ROI).  `wb_ic_roi` = within-price-quintile pooled IC vs `roi_res` — residual edge after the price axis is held roughly constant.

In [16]:
from signal_lab.stage1 import run_strategy, evaluate_signal_panel
from signal_lab.strategies import ExposureSignals
from signal_lab.signal_lib import spearman_rho

strategy = ExposureSignals(fracs=(1.0, 0.8, 0.5, 0.2))
splits, cols = run_strategy(df_full, wallet_metrics, hold_metrics, strategy)
print({s: len(splits[s]) for s in ('train', 'val', 'test')})

pooled = pd.concat(
    [splits['train'][cols + ['copyable_pnl', 'roi_res', 'price']],
     splits['val'][cols + ['copyable_pnl', 'roi_res', 'price']]],
    ignore_index=True,
)
cuts = pd.qcut(pooled['price'].rank(), 5, labels=False, duplicates='drop')
wb = {}
for c in cols:
    vals = []
    for b in cuts.dropna().unique():
        m = cuts == b
        ic = spearman_rho(pooled.loc[m, c].fillna(0.0), pooled.loc[m, 'roi_res'])
        if not np.isnan(ic):
            vals.append(ic)
    wb[c] = float(np.mean(vals)) if vals else np.nan

for target, label in (('copyable_pnl', 'pnl'), ('roi_res', 'roi')):
    report, selected = evaluate_signal_panel(splits, cols, roi_col=target)
    report['spearman_price'] = [spearman_rho(pooled[c].fillna(0.0), pooled['price']) for c in report['signal']]
    report['wb_ic_roi'] = report['signal'].map(wb)
    print(f'\n=== exposure IC vs {target} ===')
    print('selected:', selected)
    display(report[['signal', 'presence_train', 'boot_mean_ic', 'boot_ci_lo', 'boot_ci_hi',
                     'significant', 'IC_train', 'IC_val', 'IC_test', 'spearman_price', 'wb_ic_roi']])

Chronological split: train <= 2025-10-03T00:00:00Z, val <= 2026-03-25T00:00:00Z, test > 2026-03-25T00:00:00Z
Method: chronological  |  Unique end dates: 493  (train=197, val=147, test=149)

  Train:     16,007 trades  (2,119 markets)
  Val:       16,755 trades  (2,527 markets)
  Test:      10,053 trades  (1,342 markets)
  Total:     42,815 trades  (5,988 markets)
{'train': 16007, 'val': 16755, 'test': 10053}

=== exposure IC vs copyable_pnl ===
selected: ['sig_exp_pos', 'sig_exp_usdc', 'sig_exp_log', 'sig_exp_wt_rank', 'sig_exp_wt_minmax', 'sig_exp_marg', 'sig_exp_marg_wt', 'sig_exp_thr_0.2', 'sig_exp_thr_0.5', 'sig_exp_thr_0.8', 'sig_exp_top_0.8', 'sig_exp_top_0.5', 'sig_exp_top_0.2']


,signal,presence_train,boot_mean_ic,boot_ci_lo,boot_ci_hi,significant,IC_train,IC_val,IC_test,spearman_price,wb_ic_roi
0,sig_exp_wt_rank,0.997064,0.283516,0.273062,0.295070,True,0.320630,0.257148,0.120225,0.485735,-0.082728
1,sig_exp_wt_minmax,0.997064,0.263040,0.252844,0.273720,True,0.302028,0.231872,0.167625,0.468857,-0.081870
2,sig_exp_thr_0.8,0.316611,0.266701,0.255159,0.277974,True,0.300842,0.240106,0.167161,0.396223,-0.077370
3,sig_exp_log,1.000000,0.291536,0.281125,0.302315,True,0.297569,0.288488,0.234400,0.477611,-0.159192
4,sig_exp_usdc,1.000000,0.291535,0.281124,0.302315,True,0.297569,0.288488,0.234398,0.477610,-0.159191
5,sig_exp_top_0.2,0.201162,0.233681,0.223099,0.245151,True,0.265081,0.217380,0.098778,0.359563,-0.044767
6,sig_exp_top_0.5,0.500906,0.242043,0.231495,0.252436,True,0.264371,0.224578,0.129327,0.397259,-0.044095
7,sig_exp_marg,1.000000,0.262840,0.253236,0.273407,True,0.259774,0.266114,0.193389,0.507528,-0.156953
8,sig_exp_thr_0.5,0.219779,0.239596,0.228403,0.251530,True,0.258512,0.230345,0.088567,0.356730,-0.074813
9,sig_exp_marg_wt,0.997189,0.230125,0.220118,0.241093,True,0.245282,0.216887,0.104997,0.452390,-0.088706



=== exposure IC vs roi_res ===
selected: ['sig_exp_pos', 'sig_exp_usdc', 'sig_exp_log', 'sig_exp_wt_rank', 'sig_exp_wt_minmax', 'sig_exp_marg', 'sig_exp_marg_wt', 'sig_exp_thr_0.2', 'sig_exp_thr_0.5', 'sig_exp_thr_0.8', 'sig_exp_top_0.8', 'sig_exp_top_0.5', 'sig_exp_top_0.2']


,signal,presence_train,boot_mean_ic,boot_ci_lo,boot_ci_hi,significant,IC_train,IC_val,IC_test,spearman_price,wb_ic_roi
0,sig_exp_pos,1.000000,-0.144038,-0.155105,-0.132625,True,-0.130829,-0.169146,-0.133634,0.063602,-0.146767
1,sig_exp_log,1.000000,-0.105554,-0.117308,-0.094762,True,-0.095670,-0.102723,-0.035146,0.477611,-0.159192
2,sig_exp_usdc,1.000000,-0.105553,-0.117307,-0.094761,True,-0.095668,-0.102723,-0.035147,0.477610,-0.159191
3,sig_exp_marg,1.000000,-0.102021,-0.113640,-0.091313,True,-0.092583,-0.094600,-0.035501,0.507528,-0.156953
4,sig_exp_thr_0.2,0.102330,-0.048468,-0.058519,-0.038863,True,-0.075988,-0.029129,-0.024682,0.325974,-0.045008
5,sig_exp_thr_0.5,0.219779,-0.062298,-0.072960,-0.052546,True,-0.052808,-0.073809,-0.032839,0.356730,-0.074813
6,sig_exp_wt_minmax,0.997064,-0.044664,-0.054612,-0.033656,True,-0.045795,-0.035953,0.005898,0.468857,-0.081870
7,sig_exp_thr_0.8,0.316611,-0.056148,-0.067150,-0.045919,True,-0.042951,-0.067624,-0.001868,0.396223,-0.077370
8,sig_exp_marg_wt,0.997189,-0.047592,-0.059966,-0.036874,True,-0.037557,-0.042029,-0.023471,0.452390,-0.088706
9,sig_exp_top_0.2,0.201162,-0.035908,-0.045807,-0.025484,True,-0.033340,-0.041144,-0.057485,0.359563,-0.044767


Pooled IC was the wrong lens: with bankroll management, higher relative exposure should require higher *Sharpe*.  Conditional within-wallet bin means of `roi_res` (train-fit rank01) rise monotonically with exposure rank on every split, and the wallet-level `copyable_roi` Sharpe of top-20% vs the rest is positive. The first reading called this **sizing-unstable** because the constrained $10k skip-race sim flipped val↔test — but rerunning `exposure_0.5/0.8` / `top20` sizing with `--unrestricted` (budget=inf, proportional to the wallet's own book) beats `copy_all` on both roi_w and Sharpe on val *and* test, so the flip was a budget artifact, not the signal (see `exposure_bins_exposure.csv` / `exposure_wallet_summary.csv` / `exposure_sim_unrestricted.csv`).

In [17]:
from signal_lab.strategies.exposure_threshold import fit_wallet_exposure_grids, apply_wallet_rank01

splits['train']['exposure'] = splits['train']['position'] * splits['train']['price']
for s in ('val', 'test'):
    splits[s]['exposure'] = splits[s]['position'] * splits[s]['price']
grids = fit_wallet_exposure_grids(splits['train'], 'exposure')
for s in ('train', 'val', 'test'):
    fr = splits[s]
    fr['exp_rank01'] = np.nan
    for w in grids:
        m = fr['wallet'].to_numpy() == w
        if m.any():
            fr.loc[m, 'exp_rank01'] = apply_wallet_rank01(fr.loc[m, 'exposure'].to_numpy(), w, grids)

rows = []
for s in ('train', 'val', 'test'):
    fr = splits[s]
    top = fr[fr['exp_rank01'] >= 0.8]['copyable_roi']
    rest = fr[fr['exp_rank01'] < 0.8]['copyable_roi']
    rows.append({'split': s, 'mean_roi_res_top': fr[fr['exp_rank01'] >= 0.8]['roi_res'].mean(),
                 'mean_roi_res_rest': fr[fr['exp_rank01'] < 0.8]['roi_res'].mean(),
                 'copyable_roi_sharpe_top': top.mean() / top.std(),
                 'copyable_roi_sharpe_rest': rest.mean() / rest.std()})
display(pd.DataFrame(rows).set_index('split'))

,mean_roi_res_top,mean_roi_res_rest,copyable_roi_sharpe_top,copyable_roi_sharpe_rest
split,,,,
train,-0.042640,0.010738,0.425899,0.093659
val,-0.062671,0.032985,0.130228,0.047063
test,-0.058688,0.073794,0.047326,0.005264


## 15. Exposure-threshold sizing: does copying only high-exposure trades raise ROI/Sharpe?

Unrestricted sim (budget=inf, sized proportional to the wallet's own book, 10bps, depth-capped
fills at the recorded entry, PnL realized at resolution). `roi_w` = net PnL / deployed notional
(not annualized; capital is locked until resolution, median ~27 days); `sharpe_daily` = annualized
mean/std of the daily-PnL series.

- **`exposure_{frac}`** — per-wallet threshold fit on train at the exposure where cumulative PnL
  (exposure-desc) first reaches `frac × total`; copy trades with `exposure ≥ T`. Looser=1.0.
- **`exposure_top{t}`** — per-wallet train quantile: copy only the top `t` of the wallet's own book.

Key question: on **test**, does roi/Sharpe increase as we copy only the higher-exposure trades?
All numbers below are point estimates from the saved CSVs (already generated, read-only here).

In [18]:
import pandas as pd

LADDER = ['copy_all', 'exposure_1', 'exposure_0.8', 'exposure_0.5', 'exposure_0.2',
          'exposure_top80', 'exposure_top50', 'exposure_top20']

def exposure_view(path, label):
    sim = pd.read_csv(path)
    sim = sim[sim['design'].isin(LADDER)]
    df = sim.pivot_table(index='design', columns='split', values=['trades', 'pnl', 'roi_w', 'sharpe_daily'], aggfunc='first')
    df = df.reindex(LADDER)
    cols = [('trades', 'val'), ('trades', 'test'), ('pnl', 'val'), ('pnl', 'test'),
            ('roi_w', 'val'), ('roi_w', 'test'), ('sharpe_daily', 'val'), ('sharpe_daily', 'test')]
    short = {'val': 'v', 'test': 't'}
    out = pd.DataFrame({f'{m}_{short[s]}': df[(m, s)] for m, s in cols})
    base = out.loc['copy_all']
    out['test_roi_vs_copy'] = out['roi_w_t'].apply(lambda v: '>' if v > base['roi_w_t'] else '<=')
    out['test_sh_vs_copy'] = out['sharpe_daily_t'].apply(lambda v: '>' if v > base['sharpe_daily_t'] else '<=')
    out['val_roi_vs_copy'] = out['roi_w_v'].apply(lambda v: '>' if v > base['roi_w_v'] else '<=')
    out['val_sh_vs_copy'] = out['sharpe_daily_v'].apply(lambda v: '>' if v > base['sharpe_daily_v'] else '<=')
    print(f'\n=== {label} ===')
    print('  ">" = design beats copy_all on that metric/split')
    return out[['trades_v', 'roi_w_v', 'sharpe_daily_v', 'val_roi_vs_copy', 'val_sh_vs_copy',
                'trades_t', 'roi_w_t', 'sharpe_daily_t', 'test_roi_vs_copy', 'test_sh_vs_copy']].round(3)

display(exposure_view('exposure_sim_unrestricted.csv', 'Politics (full, cached splits)'))
try:
    display(exposure_view('exposure_sim_unrestricted-Weather.csv', 'Weather (2 shards)'))
except FileNotFoundError:
    print('Weather unrestricted sim not present')


=== Politics (full, cached splits) ===
  ">" = design beats copy_all on that metric/split


,trades_v,roi_w_v,sharpe_daily_v,val_roi_vs_copy,val_sh_vs_copy,trades_t,roi_w_t,sharpe_daily_t,test_roi_vs_copy,test_sh_vs_copy
design,,,,,,,,,,
copy_all,49306,0.017,0.435,<=,<=,48653,0.085,0.567,<=,<=
exposure_1,13915,0.056,1.920,>,>,9514,0.122,1.045,>,>
exposure_0.8,9780,0.031,1.539,>,>,6750,0.156,1.598,>,>
exposure_0.5,6448,0.030,1.945,>,>,4370,0.179,1.609,>,>
exposure_0.2,3161,0.015,1.179,<=,>,2542,0.187,3.328,>,>
exposure_top80,37203,0.015,0.430,<=,<=,34972,0.089,0.629,>,>
exposure_top50,21752,0.006,0.200,<=,<=,19367,0.112,0.969,>,>
exposure_top20,7851,0.024,1.119,>,>,5893,0.206,2.145,>,>



=== Weather (2 shards) ===
  ">" = design beats copy_all on that metric/split


,trades_v,roi_w_v,sharpe_daily_v,val_roi_vs_copy,val_sh_vs_copy,trades_t,roi_w_t,sharpe_daily_t,test_roi_vs_copy,test_sh_vs_copy
design,,,,,,,,,,
copy_all,10301,-0.036,-0.426,<=,<=,6573,0.028,0.230,<=,<=
exposure_1,7695,-0.042,-0.543,<=,<=,4924,0.017,0.168,<=,<=
exposure_0.8,5786,-0.042,-0.552,<=,<=,3863,0.010,0.108,<=,<=
exposure_0.5,3708,-0.040,-0.572,<=,<=,2558,0.022,0.248,<=,>
exposure_0.2,1882,-0.017,-0.222,>,>,1375,-0.012,-0.140,<=,<=
exposure_top80,8416,-0.038,-0.478,<=,<=,5473,0.031,0.263,>,>
exposure_top50,5709,-0.041,-0.532,<=,<=,3805,0.034,0.308,>,>
exposure_top20,2942,-0.061,-0.827,<=,<=,2022,0.048,0.508,>,>


**Test-panel answer (Politics):** ROI *and* Sharpe both rise with high-exposure limiting on test —
`roi_w` goes copy_all 0.085 → exposure_1 0.122 → 0.8 0.156 → 0.5 0.179 → 0.2 0.187 (top20 0.207),
and Sharpe go 0.57 → 1.05 → 1.60 → 1.61 → 3.33. The `exposure_top*` ladder (pure high-exposure tail,
no PnL-capture floor) shows the same: top80 0.089/0.63 → top50 0.112/0.97 → top20 0.207/2.15.

**Val-panel (the honest warning):** the same ladder *removes* value on val — extra concentration
beyond the 100%-capture floor hurts (exposure_1 0.0562 → 0.8 0.0307 → 0.5 0.0298 → 0.2 0.0153;
top50 is the worst design at 0.0055). So dropping the value-destroying *low*-exposure tail is
consistent across splits (copy_all → exposure_1 wins everywhere), but a high-exposure-**only** book
is val-poor / test-strong: the direction still flips at the extreme cuts, and the 7-day block-bootstrap
Sharpe CIs include zero (`exposure_ci_unrestricted.csv`). Do not read the test point estimates as a
deployable edge without out-of-sample confirmation.

Caveat on `pnl`: it is full-depth copy PnL (alpha=1, depth-capped fills at the recorded entry, PnL at
resolution, 10bps cost) — not what a fixed $10k account would earn, and not slippage-adjusted beyond 10bps.

## 16. Event-dominance & ROI diagnostics (recomputed here, read-only)

The §15 numbers re-read the saved CSVs. This cell **recomputes the key evidence in-kernel**
from `cached_splits_exposure/*.parquet`: per-wallet capture thresholds fit on **train only**, the
**unrestricted** sim (`capital_constrained_sim`, `budget=inf`, sized `alpha * qty`, depth-capped,
10bps, PnL at resolution), and the library 7-day block-bootstrap for Sharpe CIs. It answers two
questions the earlier framing was ambiguous on:

- **Q1 — event dominance:** is the val↔test difference "bad data / few events" on val?
  We report per-design, per-split top-1 / top-3 daily-PnL concentration plus the share of PnL from
  the **Iran/Hormuz question cluster** (US-Iran peace deal / ceasefire / Strait of Hormuz).
- **Q2 — does exposure correlate with trade ROI, and do we need to normalize?**
  The §14 IC panel shows the *continuous* signal is ~0/negative vs `roi_res` on val/test, but the
  binary *tail* (`thr_0.8`, `top_0.2`) is positively significant on val. Here we add a
  within-wallet exposure-decile table (roi_w / mean `roi_res` / pct_pos per split) to see whether
  the effect is a monotone ranking or a tail/risk carve, and an **Iran-cluster-exclusion control**
  on test to separate the cluster from the carve.

All numbers in this cell recompute from the parquet files — the test rows below should match
`exposure_ci_unrestricted.csv` exactly (sanity row printed).

In [19]:
import sys as _sys, numpy as np, pandas as pd
from signal_lab.sizing import capital_constrained_sim, block_bootstrap_sharpe, sizing_sharpe
from signal_lab.strategies.exposure_threshold import wallet_capture_thresholds

LADDER = ['copy_all', 'exposure_1', 'exposure_0.8', 'exposure_0.5', 'exposure_0.2']
THE_FRAC = {'exposure_1': 1.0, 'exposure_0.8': 0.8, 'exposure_0.5': 0.5, 'exposure_0.2': 0.2}

def load_split(split):
    fr = pd.read_parquet(f'cached_splits_exposure/{split}.parquet')
    fr['exposure'] = fr['position'] * fr['price']
    fr['score1'] = 1.0
    fr['bucket_avail_copy_qty'] = fr['avail_copy_qty_5m_100'].clip(lower=0.0).fillna(fr['copyable_qty_5m_100'])
    return fr

train = load_split('train')
th_full = wallet_capture_thresholds(train, (1.0, 0.8, 0.5, 0.2), 'copyable_pnl', 'exposure')
thmap = {f: th_full[th_full['frac'] == f].set_index('wallet')['threshold'] for f in (1.0, 0.8, 0.5, 0.2)}

def run_sim(split, design, drop_iran=False):
    fr = load_split(split)
    if drop_iran:
        fr = fr.loc[~fr['question'].fillna('').str.contains('Iran|Hormuz', case=False, regex=True)].copy()
    if design != 'copy_all':
        T = fr['wallet'].map(thmap[THE_FRAC[design]])
        fr = fr[(fr['exposure'] >= T).fillna(False)].copy()
    fr['alpha_w'] = 1.0
    return capital_constrained_sim(fr, 'score1', float('inf'), 1.0, cost_bps=10.0,
                                   alpha_col='alpha_w', cap_col='bucket_avail_copy_qty')

def sim_row(split, design, drop_iran=False):
    res = run_sim(split, design, drop_iran)
    p, lo, hi = block_bootstrap_sharpe(res['daily_pnl'], block_size=7, n_iter=1000, seed=42)
    return {
        'trades': res['trades'],
        'pnl': round(res['net_pnl']),
        'roi_w': round(res['net_pnl'] / res['notional'], 4) if res['notional'] else 0.0,
        'sharpe_daily': round(sizing_sharpe(res['daily_pnl']), 3),
        'ci_lo': round(lo, 3), 'ci_hi': round(hi, 3),
    }

# --- Q1: event dominance (top-day concentration + Iran/Hormuz cluster share) ---
rows = []
for split in ('val', 'test'):
    for design in LADDER:
        res = run_sim(split, design)
        daily = res['daily_pnl'].sort_values(ascending=False)
        tot = daily.sum()
        rows.append({
            'split': split, 'design': design,
            'trades': res['trades'],
            'top1_day_share': round(float(daily.iloc[0] / tot), 4) if tot != 0 else 0.0,
            'top3_day_share': round(float(daily.head(3).sum() / tot), 4) if tot != 0 else 0.0,
            'pnl': round(res['net_pnl']),
        })
ed = pd.DataFrame(rows).set_index(['split', 'design'])
print('Top-1 / top-3 trading-day concentration of sim PnL (a 1.0 top-1 share = single-day book).')
display(ed)

def iran_cluster_share():
    out = []
    for split in ('val', 'test'):
        fr = load_split(split)
        for design in LADDER:
            f = fr if design == 'copy_all' else fr[(fr['exposure'] >= fr['wallet'].map(thmap[THE_FRAC[design]])).fillna(False)]
            iran = f['copyable_pnl'].loc[f['question'].fillna('').str.contains('Iran|Hormuz', case=False, regex=True)].sum()
            out.append({'split': split, 'design': design,
                        'iran_share_of_pnl': round(float(iran / f['copyable_pnl'].sum()), 4) if f['copyable_pnl'].sum() != 0 else 0.0})
    return pd.DataFrame(out).set_index(['split', 'design'])

ics = iran_cluster_share()
print('\nShare of each design\'s gross copyable PnL from the Iran/Hormuz question cluster.')
display(ics)

# --- Q2a: does the exposure carve survive removing the dominant cluster? (test control) ---
ctrl = pd.DataFrame([
    {'drop_iran': d, **{f'{m}_roi_w': sim_row('test', m, d)['roi_w'] for m in LADDER},
     **{f'{m}_sharpe': sim_row('test', m, d)['sharpe_daily'] for m in LADDER}}
    for d in (False, True)
]).set_index('drop_iran')
ctrl.index = ['full_test', 'test_minus_iran']
print('\nIran-exclusion control on test: roi_w (then Sharpe) per design.')
display(ctrl)

# --- Q2b: within-wallet exposure-decile ROI (is exposure a monotone ranking or a tail carve?) ---
grd = {w: np.sort(g['exposure'].to_numpy(dtype=float)) for w, g in train.groupby('wallet')}

def wrank_01(vals, grid):
    left = np.searchsorted(grid, vals, side='left')
    right = np.searchsorted(grid, vals, side='right')
    return (left + right + 1.0) / 2.0 / (len(grid) - 1.0)

drows = []
for split in ('train', 'val', 'test'):
    fr = load_split(split)
    w = fr['wallet'].to_numpy()
    rk = np.full(len(fr), np.nan)
    for wal in np.unique(w):
        m = np.where(w == wal)[0]
        g = grd.get(wal)
        if g is not None and len(g):
            rk[m] = wrank_01(fr['exposure'].to_numpy(dtype=float)[m], g)
    fr = fr.assign(_rk=rk).dropna(subset=['_rk']).copy()
    fr['_dec'] = (fr['_rk'] * 10).clip(0, 9).astype(int)
    g = fr.groupby('_dec').agg(
        n=('copyable_pnl', 'count'),
        roi_w=('copyable_pnl', lambda v: v.sum() / fr.loc[v.index, 'copyable_notional'].sum()),
        mean_roi_res=('roi_res', 'mean'),
        pct_pos=('copyable_pnl', lambda v: (v > 0).mean()),
    )
    g['split'] = split
    drows.append(g.reset_index())
dec = pd.concat(drows, ignore_index=True)
dec['roi_w'] = dec['roi_w'].round(4)
dec['mean_roi_res'] = dec['mean_roi_res'].round(4)
dec['pct_pos'] = dec['pct_pos'].round(3)
print('\nWithin-wallet exposure-rank decile, out-of-sample (train-fit grid on val/test).')
print('roi_w by split:')
display(dec.pivot_table(index='_dec', columns='split', values='roi_w'))
print('mean roi_res by split:')
display(dec.pivot_table(index='_dec', columns='split', values='mean_roi_res'))
print('pct_pos (gross PnL > 0) by split:')
display(dec.pivot_table(index='_dec', columns='split', values='pct_pos'))

# --- sanity: authoritative val+test CIs (should match the saved test CI file on test) ---
ci_rows = [{'split': s, 'design': d, **sim_row(s, d)} for s in ('val', 'test') for d in LADDER]
ci = pd.DataFrame(ci_rows)
print('\nAuthoritative val+test Sharpe CIs (block bootstrap, 7-day, seed 42):')
display(ci.set_index(['split', 'design']).round(3))
sanity = ci[(ci['split'] == 'test') & (ci['design'] == 'exposure_0.8')].iloc[0]
saved = pd.read_csv('exposure_ci_unrestricted.csv')
saved = saved[(saved['cost_bps'] == 10.0) & (saved['design'] == 'exposure_0.8')].iloc[0]
print(f"Sanity: recomputed test exposure_0.8 roi_w={sanity['roi_w']}, sharpe={sanity['sharpe_daily']}, CI=[{sanity['ci_lo']},{sanity['ci_hi']}] "
      f"=> saved file roi_w={saved['roi_w']}, sharpe={saved['sharpe_daily']}, CI=[{saved['ci_lo']},{saved['ci_hi']}]")

Top-1 / top-3 trading-day concentration of sim PnL (a 1.0 top-1 share = single-day book).


trades  top1_day_share  top3_day_share     pnl
split design                                                      
val   copy_all       49306          0.4090          1.1400   41411
      exposure_1     13915          0.1934          0.4269  103244
      exposure_0.8    9780          0.1920          0.4573   51926
      exposure_0.5    6448          0.1442          0.3850   46535
      exposure_0.2    3161          0.3490          0.8659   16486
test  copy_all       48653          0.6917          1.4674  180446
      exposure_1      9514          0.6255          1.0918  170908
      exposure_0.8    6750          0.5018          0.8665  184191
      exposure_0.5    4370          0.6007          0.8368  150417
      exposure_0.2    2542          0.2395          0.5596   78005


Share of each design's gross copyable PnL from the Iran/Hormuz question cluster.


iran_share_of_pnl
split design                         
val   copy_all                 0.7659
      exposure_1               0.4846
      exposure_0.8             0.4317
      exposure_0.5             0.4543
      exposure_0.2             0.0036
test  copy_all                 1.3194
      exposure_1               1.0480
      exposure_0.8             1.0242
      exposure_0.5             0.9429
      exposure_0.2             0.7867


Iran-exclusion control on test: roi_w (then Sharpe) per design.


...

pct_pos (gross PnL > 0) by split:


split,test,train,val
_dec,,,
0,0.316,0.162,0.245
1,0.223,0.201,0.205
2,0.240,0.235,0.217
3,0.252,0.293,0.249
4,0.298,0.325,0.312
5,0.352,0.363,0.313
6,0.396,0.382,0.327
7,0.352,0.435,0.314
8,0.457,0.527,0.519



Authoritative val+test Sharpe CIs (block bootstrap, 7-day, seed 42):


trades     pnl  roi_w  sharpe_daily  ci_lo  ci_hi
split design                                                         
val   copy_all       49306   41411  0.017         0.435 -2.402  2.507
      exposure_1     13915  103244  0.056         1.920  1.565  4.139
      exposure_0.8    9780   51926  0.031         1.539  1.219  4.728
      exposure_0.5    6448   46535  0.030         1.945  1.777  5.545
      exposure_0.2    3161   16486  0.015         1.179 -1.712  4.329
test  copy_all       48653  180446  0.085         0.567 -2.412  1.886
      exposure_1      9514  170908  0.122         1.045 -2.750  1.597
      exposure_0.8    6750  184191  0.156         1.598 -2.017  2.247
      exposure_0.5    4370  150417  0.179         1.609 -0.655  2.498
      exposure_0.2    2542   78005  0.187         3.328 -0.297  2.783

Sanity: recomputed test exposure_0.8 roi_w=0.1563, sharpe=1.598, CI=[-2.017,2.247] => saved file roi_w=0.1563, sharpe=1.598, CI=[-2.017,2.247]


**What this recompute changes:**

1. **"Bad data on val?" — no; the event-dominance is on *test*.** Val's top-1 day is only ~0.2-0.4 of PnL
   (vs test ~0.5-0.7) and the gap is concentration *level* driven: the **Iran/Hormuz cluster is 132% of
   test copy_all PnL** (non-Iran book nets negative on test) vs 77% on val. Val is a well-spread panel — the
   residual val↔test flip is not a "few events on val" artifact.
2. **The interesting caveat cuts the other way:** the exposure *carve* is **not** a cluster artifact either.
   Delete Iran/Hormuz from test entirely and `copy_all` goes negative (roi_w -0.10, Sharpe -0.66) while
   `exposure_0.5` and `exposure_0.2` stay positive (+0.024/+0.104 roi_w, Sharpe +0.61/+4.04). The filter's
   advantage survives removing the dominant cluster — it is not a restatement of "Iran was good".
3. **Exposure is a tail/risk carve, not a monotone ROI ranking.** Within-wallet exposure-decile `roi_w` is
   non-monotone (the middle deciles are the worst; val dec-0 "lottery" low-exposure winners inflate the lowest
   bucket), but **pct_pos rises monotonically with decile** on every split (val 0.24 → 0.51; test 0.32 → 0.48),
   and mean `roi_res` is positive at the top. This is *why* the continuous exposure IC vs `roi_res` is ~0/
   negative while the binary PnL-capture tail (`thr_0.8`, `top_0.2`) is positively significant on val: the
   message is "drop the value-destroying low-exposure mass", which is exactly the `exposure_{0.8,0.5}` /
   `top20` carve that survives the Iran control. We **do need the within-wallet construction** (raw exposure
   is dominated by wallet size/price, skew ≈ 11-23) — but normalization alone is not enough; the exploitable
   form is the per-wallet PnL-informed tail, not `rank01(exposure)` as a monotone signal.

**Bottom line (whole §15-16):** copy the high-exposure tail, don't read it as a funding/ranking dial. It is a
*risk carve* — it removes the loss-making low-exposure mass and concentrates the book on trades whose win
rate rises monotonically with within-wallet exposure. Robust: consistent val+test direction for
`exposure_{0.8,0.5}`/`top20`, survives Iran-cluster removal, val CIs exclude zero. Not yet done: the effect
is small on the 2-shard Weather val, and the extreme cut (`exposure_0.2`) still flips on val — keep the
100%-capture floor as the usable range and get fresh out-of-sample before deployment.

## 17. Volatility-neutral target (`z_vol`): is the exposure edge "profit per unit vol"?

Discussion developed: to *"maximize profit per volatility"* the right trade target is a **studentized ROI** —

```
z_vol = (copyable_roi − μ_train(price)) / sd_train(price)      (per price octile, train-fit only)
```

`roi_res` (rank-normalized, §14-16) removes only the *mean* price effect. `z_vol` removes the *mean and the
scale* — it is the per-trade "edge per unit of price-typical risk" (profit-per-vol) target. This cell recomputes
the §14-16 exposure diagnostics against `z_vol` to answer: **does the copy-wallet exposure carve survive when
you charge for the fact that cheap trades are inherently more volatile?**

Because cheap-price SD is jackpot-dominated (below: `sd_clip5` vs `sd_raw` on the cheap octile), the scale is fit
on **train only**; nothing val/test touches the thresholds.

Questions:

- **Q1 — de-confounding:** does `z_vol` actually make `rho(price, ·)` ~ 0 (raw roi is +0.18/+0.36)?
- **Q2 — carve survival:** does the high-exposure tail still beat the low tail within price bins under `z_vol`?
- **Q3 — objective split:** does the exposure filter raise mean `z_vol` / risk-neutral PnL of the copied book?

In [20]:
import numpy as np, pandas as pd
from signal_lab.strategies.exposure_threshold import (
    wallet_capture_thresholds, fit_wallet_exposure_grids, apply_wallet_rank01)
from signal_lab.signal_lib import spearman_rho

LADDER = ['copy_all', 'exposure_1', 'exposure_0.8', 'exposure_0.5', 'exposure_0.2']
THE_FRAC = {'exposure_1': 1.0, 'exposure_0.8': 0.8, 'exposure_0.5': 0.5, 'exposure_0.2': 0.2}
BINS = 8  # price octiles

def load_split(split):
    fr = pd.read_parquet(f'cached_splits_exposure/{split}.parquet')
    fr['exposure'] = fr['position'] * fr['price']
    fr['score1'] = 1.0
    fr['bucket_avail_copy_qty'] = fr['avail_copy_qty_5m_100'].clip(lower=0.0).fillna(fr['copyable_qty_5m_100'])
    return fr

train = load_split('train')
th_all = wallet_capture_thresholds(train, (1.0, 0.8, 0.5, 0.2), 'copyable_pnl', 'exposure')
thmap = {f: th_all[th_all['frac'] == f].set_index('wallet')['threshold'] for f in (1.0, 0.8, 0.5, 0.2)}

# --- train-fit price bins and per-bin mu/sd of RAW copyable_roi (train only) ---
tt = train[train['copyable_roi'].notna() & train['roi_res'].notna()].copy()
tt['q'] = pd.qcut(tt['price'].rank(method='first'), BINS, labels=False, duplicates='drop')
MU = tt.groupby('q', observed=True)['copyable_roi'].mean()
SD = tt.groupby('q', observed=True)['copyable_roi'].std()
sd_clip5 = tt.assign(_c=tt['copyable_roi'].clip(upper=5.0)).groupby('q', observed=True)['_c'].std()
print('\nTrain-fit mu/sd of raw copyable_roi by price bin (sd_clip5 = jackpot-sensitivity of the scale):')
display(pd.DataFrame({'mu_raw': MU, 'sd_raw': SD, 'sd_clip5': sd_clip5}).round(3))

def add_z(fr):
    fr = fr[fr['copyable_roi'].notna() & fr['roi_res'].notna()].copy()
    fr['q'] = pd.qcut(fr['price'].rank(method='first'), BINS, labels=False, duplicates='drop')
    mu = MU.reindex(fr['q']).to_numpy(); sd = SD.reindex(fr['q']).to_numpy()
    fr['z_vol'] = (fr['copyable_roi'] - mu) / np.maximum(sd, 1e-9)
    return fr

# --- Q1: does z_vol actually de-confound price? ---
dc_rows = []
for split in ('train', 'val', 'test'):
    fr = add_z(load_split(split))
    dc_rows.append({'split': split,
                    'rho_price_raw': round(spearman_rho(fr['copyable_roi'], fr['price']), 3),
                    'rho_price_roi_res': round(spearman_rho(fr['roi_res'], fr['price']), 3),
                    'rho_price_zvol': round(spearman_rho(fr['z_vol'], fr['price']), 3)})
print('\nSpearman rho(price, target): z_vol should be ~0 (both mean and scale removed).')
display(pd.DataFrame(dc_rows))

# --- Q2: does the exposure carve survive under the vol-neutral target? ---
grids = fit_wallet_exposure_grids(train, 'exposure')
trows = []
for split in ('train', 'val', 'test'):
    fr = add_z(load_split(split))
    fr['rank_exp'] = np.nan
    for w, g in fr.groupby('wallet', sort=False):
        if w in grids:
            fr.loc[g.index, 'rank_exp'] = apply_wallet_rank01(g['exposure'].to_numpy(float), w, grids)
    fr = fr.dropna(subset=['rank_exp'])
    for measure, col in [('roi_res', 'roi_res'), ('z_vol', 'z_vol')]:
        deltas, npos, nq = [], 0, 0
        for q, seg in fr.groupby('q', observed=True):
            top = seg[seg['rank_exp'] >= 0.8]; bot = seg[seg['rank_exp'] < 0.2]
            if len(top) >= 5 and len(bot) >= 5:
                d = float(top[col].mean() - bot[col].mean())
                deltas.append(d); nq += 1; npos += int(d > 0)
        trows.append({'split': split, 'measure': measure, 'n_bins': nq,
                      'n_top_gt_bot': npos, 'mean_delta': round(np.mean(deltas), 4) if deltas else np.nan})
print('\nExposure tail carve (top-20% vs bottom-20% within-wallet rank) within price bin.')
print('n_top_gt_bot = # price bins where the high-exposure tail has higher mean target.')
display(pd.DataFrame(trows))

# --- Q3: filter-level vol-neutral performance ---
frows = []
for split in ('train', 'val', 'test'):
    fr = add_z(load_split(split))
    for design in LADDER:
        f = fr if design == 'copy_all' else fr[
            (fr['exposure'] >= fr['wallet'].map(thmap[THE_FRAC[design]])).fillna(False)]
        frows.append({'split': split, 'design': design, 'n': len(f),
                      'mean_z_vol': round(float(f['z_vol'].mean()), 4),
                      'risk_pnl': round(float((f['copyable_pnl'] / np.maximum(SD.reindex(f['q']).to_numpy(), 1e-9)).sum())),
                      'pnl': round(float(f['copyable_pnl'].sum()))})
ft = pd.DataFrame(frows)
print('\nMean z_vol of the copied set (higher = more profit per unit of price-typical risk).')
display(ft.pivot_table(index='design', columns='split', values='mean_z_vol').round(3))
print('Risk-neutral PnL = sum(copyable_pnl / sd_train(price_bin)).')
display(ft.pivot_table(index='design', columns='split', values='risk_pnl').round(0))
print('Total copyable PnL (raw $, for comparison).')
display(ft.pivot_table(index='design', columns='split', values='pnl').round(0))


Train-fit mu/sd of raw copyable_roi by price bin (sd_clip5 = jackpot-sensitivity of the scale):


,mu_raw,sd_raw,sd_clip5
q,,,
0,0.930,9.486,1.766
1,0.552,2.452,2.345
2,0.600,1.650,1.650
3,0.429,1.194,1.194
4,0.193,0.864,0.864
5,0.187,0.613,0.613
6,0.074,0.436,0.436
7,0.020,0.179,0.179



Spearman rho(price, target): z_vol should be ~0 (both mean and scale removed).


,split,rho_price_raw,rho_price_roi_res,rho_price_zvol
0,train,0.180,0.047,0.069
1,val,0.361,0.100,0.123
2,test,0.303,-0.006,0.071



Exposure tail carve (top-20% vs bottom-20% within-wallet rank) within price bin.
n_top_gt_bot = # price bins where the high-exposure tail has higher mean target.


...

split,test,train,val
design,,,
copy_all,-0.158,0.000,-0.181
exposure_0.2,0.046,0.255,-0.077
exposure_0.5,0.026,0.224,0.000
exposure_0.8,0.028,0.196,0.003
exposure_1,-0.013,0.163,-0.009


Risk-neutral PnL = sum(copyable_pnl / sd_train(price_bin)).


split,test,train,val
design,,,
copy_all,180239.0,774798.0,217163.0
exposure_0.2,119628.0,277729.0,40083.0
exposure_0.5,195911.0,521387.0,152892.0
exposure_0.8,194124.0,716456.0,177498.0
exposure_1,209347.0,796212.0,234858.0


Total copyable PnL (raw $, for comparison).


split,test,train,val
design,,,
copy_all,182571.0,687634.0,43861.0
exposure_0.2,78421.0,183416.0,17566.0
exposure_0.5,151256.0,430507.0,48095.0
exposure_0.8,185370.0,588142.0,53616.0
exposure_1,172306.0,695188.0,105080.0


**§17 answer — the vol-neutral target changes *what* exposure predicts, not the carve's existence.**

1. **`z_vol = (copyable_roi − μ_train(price)) / sd_train(price)` is the correct "profit per unit price-typical risk"
   target.** Fit per price octile on train only. It de-confounds price on *both* axes: `rho(price, ·)` drops from
   **+0.18/+0.36 (raw roi)** and **+0.05/+0.10 (roi_res)** to **+0.07/+0.12 (z_vol)**.
2. **The exposure carve survives vol-normalization — roughly doubled, not killed.** High- vs low-exposure tail
   deltas under `z_vol` are the same sign and ~2x the `roi_res` magnitude on every split, and the high tail still
   beats the low tail in **6–8/8 price bins** (val 8/8). The earlier "vol-normalization destroys the signal" reading
   was an artifact of a collapsed MAD scale, not the data.
3. **The carve *raises* mean z_vol of the copied book vs copy_all.** On test copy_all sits at **−0.16** (negative
   profit/vol!) and `exposure_{0.8,0.5,0.2}` lift it to **+0.03…+0.05**; on val copy_all is −0.18 and the carve
   brings it to ~0. The filter concentrates the book on trades whose edge-per-unit-risk is positive where the
   unfiltered book's is negative.
4. **Where the two objectives still disagree — the wallet-selection vs carve split:** under risk-neutral PnL,
   `exposure_1` (≈ wallet-selection: copy everything the profitable-train wallets do) is the best val *and* test
   design (risk_pnl 235k/209k vs copy_all 217k/180k). That is a *wallet-quality* effect, orthogonal to the
   exposure carve. The carve proper (`0.8/0.5`) improves risk-neutral PnL on test but not on val — the doc's
   "still small on val/Weather; keep the 1.0 floor as deployment range" caveat survives unchanged.

**Takeaway:** normalizing by price-typical volatility answers exactly your question — under "maximize profit per
unit vol", the exposure **tail** (drop the low-exposure mass) remains a genuine, vol-neutral edge, while the
*ranking* reading is still flat. Use `z_vol` as the go target when the objective is risk-adjusted; use `copyable_pnl`
as-is when it is total PnL with unlimited capital.

## 18. Exposure-filter evaluation 80/50/20/10 (the bankroll-discipline ladder)

Does the exposure edge survive at the classic bankroll cutoffs? For each design we **fit per-wallet
PnL-capture thresholds on train only** (`frac` of the wallet's train copyable PnL, exposure-desc: 0.8, 0.5,
0.2, 0.1 — i.e. copy only trades whose exposure covers the top 80/50/20/10% of the wallet's edge), then run the
**unrestricted** sim (`budget=inf`, sized `alpha * qty`, depth-capped fills, 10bps, PnL at resolution) on val
and test.

Reported per design & split:

- **trades** — how many candidate trades survive the filter
- **wallet_pnl** — the copied wallets' **realized** PnL on the taken trades (`Σ pnl`; what the signal side earned)
- **copyable_pnl** — copyable-source PnL (`Σ copyable_pnl` on the taken trades)
- **copyable_roi** — mean copyable ROI of the taken trades (per-trade, not notional-weighted)
- **notional** — total traded notional deployed (Σ qty×price)
- **pnl** — net copy PnL (depth-capped, 10bps cost)
- **roi_w** — net PnL / total traded notional
- **sharpe_daily** — annualized mean/std of the daily-PnL series (7-day block bootstrap 95% CI)

An **Iran/Hormuz-exclusion control** on test repeats the ladder without the dominant question cluster, to
confirm the filter is not a restatement of "Iran was good".


In [ ]:
import sys as _sys, numpy as np, pandas as pd
from signal_lab.sizing import capital_constrained_sim, block_bootstrap_sharpe, sizing_sharpe
from signal_lab.strategies.exposure_threshold import wallet_capture_thresholds

LADDER = ['copy_all', 'exposure_0.8', 'exposure_0.5', 'exposure_0.2', 'exposure_0.1']
THE_FRAC = {'exposure_0.8': 0.8, 'exposure_0.5': 0.5, 'exposure_0.2': 0.2, 'exposure_0.1': 0.1}

def load_split(split):
    fr = pd.read_parquet(f'cached_splits_exposure/{split}.parquet')
    fr['exposure'] = fr['position'] * fr['price']
    fr['score1'] = 1.0
    fr['bucket_avail_copy_qty'] = fr['avail_copy_qty_5m_100'].clip(lower=0.0).fillna(fr['copyable_qty_5m_100'])
    return fr

train = load_split('train')
th_full = wallet_capture_thresholds(train, (0.8, 0.5, 0.2, 0.1), 'copyable_pnl', 'exposure')
thmap = {f: th_full[th_full['frac'] == f].set_index('wallet')['threshold'] for f in (0.8, 0.5, 0.2, 0.1)}

def run_sim(split, design, drop_iran=False):
    fr = load_split(split)
    if drop_iran:
        fr = fr.loc[~fr['question'].fillna('').str.contains('Iran|Hormuz', case=False, regex=True)].copy()
    if design != 'copy_all':
        T = fr['wallet'].map(thmap[THE_FRAC[design]])
        fr = fr[(fr['exposure'] >= T).fillna(False)].copy()
    fr['alpha_w'] = 1.0
    return capital_constrained_sim(fr, 'score1', float('inf'), 1.0, cost_bps=10.0,
                                   alpha_col='alpha_w', cap_col='bucket_avail_copy_qty')

def sim_row(split, design, drop_iran=False):
    fr = load_split(split)
    if drop_iran:
        fr = fr.loc[~fr['question'].fillna('').str.contains('Iran|Hormuz', case=False, regex=True)].copy()
    if design != 'copy_all':
        T = fr['wallet'].map(thmap[THE_FRAC[design]])
        fr = fr[(fr['exposure'] >= T).fillna(False)].copy()
    fr['alpha_w'] = 1.0
    res = capital_constrained_sim(fr, 'score1', float('inf'), 1.0, cost_bps=10.0,
                                  alpha_col='alpha_w', cap_col='bucket_avail_copy_qty')
    taken = fr.loc[res['taken']] if len(res['taken']) else fr.iloc[0:0]
    p, lo, hi = block_bootstrap_sharpe(res['daily_pnl'], block_size=7, n_iter=1000, seed=42)
    return {
        'trades': res['trades'],
        'wallet_pnl': round(taken['pnl'].sum()),
        'copyable_pnl': round(taken['copyable_pnl'].sum()),
        'copyable_roi': round(taken['copyable_roi'].mean(), 4) if len(taken) else 0.0,
        'notional': round(res['notional']),
        'pnl': round(res['net_pnl']),
        'roi_w': round(res['net_pnl'] / res['notional'], 4) if res['notional'] else 0.0,
        'sharpe_daily': round(sizing_sharpe(res['daily_pnl']), 3),
        'ci_lo': round(lo, 3), 'ci_hi': round(hi, 3),
    }

rows = []
for split in ('val', 'test'):
    for design in LADDER:
        rows.append({'split': split, 'design': design, **sim_row(split, design)})
out = pd.DataFrame(rows).pivot_table(index='design', columns='split', values=['trades', 'wallet_pnl', 'copyable_pnl', 'copyable_roi', 'notional', 'pnl', 'roi_w', 'sharpe_daily', 'ci_lo', 'ci_hi'], aggfunc='first')
out = out.reindex(LADDER)

cols = []
for split in ('val', 'test'):
    for m in ('trades', 'wallet_pnl', 'copyable_pnl', 'copyable_roi', 'notional', 'pnl', 'roi_w', 'sharpe_daily'):
        cols.append((m, split))
cols += [('ci_lo', 'val'), ('ci_hi', 'val'), ('ci_lo', 'test'), ('ci_hi', 'test')]

print('\n=== Exposure-filter ladder 80/50/20/10 (unrestricted sim, thresholds fit on train) ===')
print('wallet_pnl = the copied wallets\' realized pnl on the taken trades; copyable_pnl/copyable_roi = the copy-side source')
print('notional = total traded notional; roi_w = net pnl / notional; sharpe_daily = annualized daily-PnL mean/std')
print()
sh = out[cols].round(4)
sh.columns = ['_'.join(c) for c in sh.columns]
display(sh)

# --- Iran-cluster exclusion control on test ---
ir = pd.DataFrame([{'design': d, 'split': s, **sim_row(s, d, drop_iran=True)}
                   for s in ('test',) for d in LADDER]).set_index('design')
print('\n=== Control: same ladder on test with Iran/Hormuz question cluster excluded ===')
display(ir[['trades', 'wallet_pnl', 'copyable_pnl', 'copyable_roi', 'notional', 'pnl', 'roi_w', 'sharpe_daily']].round(4))

**§18 answer — the exposure carve survives at 80/50/20/10; the extreme cuts trade on the val↔test asymmetry.**

**The ladder (unrestricted sim, thresholds fit on train only):**

| | val: wallet_pnl | copyable_pnl | copyable_roi | test: wallet_pnl | copyable_pnl | copyable_roi | notional |
|---|---|---|---|---|---|---|---|
| copy_all | 323,128 | 43,861 | 0.034 | 299,681 | 182,571 | 0.071 | 2.12M |
| exposure_0.8 | 189,737 | 53,616 | 0.149 | 312,328 | 185,370 | 0.291 | 1.18M |
| exposure_0.5 | 169,598 | 48,095 | 0.140 | 269,611 | 151,256 | 0.426 | 0.84M |
| exposure_0.2 | 62,230 | 17,566 | 0.068 | 174,052 | 78,421 | 0.405 | 0.42M |
| exposure_0.1 | 15,963 | 5,107 | 0.009 | 130,206 | 57,016 | 0.174 | 0.35M |

**Iran/Hormuz-excluded test control:**

| | wallet_pnl | copyable_pnl | copyable_roi | roi_w | Sharpe CI |
|---|---|---|---|---|---|
| copy_all | −80,645 | −58,320 | −0.213 | −0.1008 | [−1.48, 2.32] |
| exposure_0.8 | −2,671 | −4,483 | 0.031 | −0.0123 | [−1.96, 2.52] |
| exposure_0.5 | 26,040 | 8,636 | 0.026 | 0.0244 | [−1.86, 2.63] |
| exposure_0.2 | 35,512 | 16,728 | −0.071 | 0.1043 | **[0.92, 3.34]** |
| exposure_0.1 | 27,635 | 12,705 | −0.067 | 0.0976 | **[1.25, 3.50]** |

**Reading it straight:**

1. **The 80/50 carve is the robust core (and the signal source confirms it).** On val *and* test the retained
   trades carry **higher wallet-pnl per trade and higher copyable_roi per trade** than copy_all (val copyable_roi
   0.034 → 0.149/0.140; test 0.071 → 0.291/0.426) — the high-exposure tail really is where the copied wallets'
   edge lives, not just a copy-side accounting artifact. On test the 0.8 carve even keeps *more* wallet_pnl and
   copyable_pnl than copy_all (312k/185k vs 300k/183k) while copying only 6,750 of 48,653 trades.
2. **The 20/10 extreme cuts keep the §15-16 asymmetry.** Test roi_w *keeps rising* (0.187, 0.162) and Sharpe
   triples; val roi_w collapses back toward copy_all (0.015, 0.005). The signal source mirrors this: val
   copyable_roi at 0.2/0.1 falls back below the 0.8/0.5 levels (0.068, 0.009) even though test source ROI stays
   high (0.405, 0.174) — the extreme tail is val-weak / test-strong.
3. **The 10% cut is *not* a test artifact.** With Iran/Hormuz excluded, copy_all nets **−0.101** while 0.2/0.1
   stay **+0.104 / +0.098** with Sharpe CIs excluding zero ([0.92, 3.34] / [1.25, 3.50]) — the only designs
   statistically distinct from zero on test. Note the source metrics flip sign (copyable_roi −0.071/−0.067:
   the Iran-excluded extreme tail still makes money in *notional-weighted* copy PnL, but its per-trade mean ROI
   is negative — a handful of large winning positions carry it). The tail is genuinely robust to the cluster;
   its weakness is specifically on *val*.
4. **Use it as a carve, not a dial.** `exposure_{0.8,0.5}` is the defensible production cut (positive both
   splits, CIs exclude zero on val, source ROI roughly 4-6× copy_all). 0.2/0.1 stay research-view until fresh
   out-of-sample data settles which split's direction is real; the filter never *reduces* roi_w below copy_all
   on test — the risk is concentration, not dilution.